# levelset_metrology: interactive exploration

Python bindings for `levelset_metrology`'s measurement primitives
(trilinear-interpolation crossings, mesh-triangle ray intersections),
scoped deliberately to `common_geometry` + this project's own
functions only -- no dependency on any extraction algorithm
(`levelset2d_polygon`, `levelset3d_polygon`, `rectilinear2d_boolean`).

Run `uv pip install -e ".[notebook]"` from the repo root first, then
launch this notebook with `jupyter lab python/notebooks/`.


In [ ]:
import numpy as np
import levelset_metrology as lm

lm.__name__, [n for n in dir(lm) if not n.startswith("_")]

## 1. Primitives: trilinear crossings and mesh-ray intersection

`trilinear_value` samples the trilinear interpolant of 8 cube-corner
values at any point in the unit cube; `find_trilinear_crossings` walks
a probe line and finds where that interpolant crosses zero.
`ray_triangle_intersect` / `find_mesh_crossings` do the analogous thing
against an explicit triangle mesh (Moller-Trumbore).


In [ ]:
v = [-1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
origin = np.array([0.0, 0.3, 0.7])
direction = np.array([1.0, 0.0, 0.0])

ts = lm.find_trilinear_crossings(v, origin, direction)
print(f"trilinear crossings: {ts}")

mesh = lm.Mesh3d(
    [np.array([0.0, 0.0, 0.0]), np.array([2.0, 0.0, 0.0]), np.array([0.0, 2.0, 0.0])],
    [(0, 1, 2)],
)
hits = lm.find_mesh_crossings(mesh, np.array([0.5, 0.5, -5.0]), np.array([0.0, 0.0, 10.0]))
print(f"mesh crossings: {hits}")

lm.plotting.plot_mesh3d(mesh)

## 2. The saddle-case mesh vs. trilinear comparison, interactively

This reproduces `levelset3d_polygon/analysis/saddle_intersection_analysis.cpp`
inside the notebook: for the classic marching-cubes face saddle (case 5 --
corners 0 and 2 of one face inside, corners 1 and 3 outside), a probe
line runs between the cube's two "inside" corners. Either method
reporting *any* crossings along the way means that method thinks the
material pinches off between them.

Since this project deliberately does not depend on `levelset3d_polygon`
(no marching cubes here), the demonstration mesh is built by
`known_cases.case5_mesh`, which hand-reconstructs the exact same fixed
two-triangle triangulation the real marching-cubes table always emits
for this case -- verified to reproduce the C++ reference crossing
values exactly (see `python/tests/test_bindings.py`).

Drag the slider below to sweep the inside corners' magnitude `s`
(`v0 = v2 = -s`, outside corners fixed at `1.0`) across the
asymptotic-decider threshold at `s = 1`, and watch the mesh (red) vs.
trilinear (green) crossing points diverge -- including the genuine
topology disagreement once `s` crosses 1.


In [ ]:
import ipywidgets as widgets
from IPython.display import display


def saddle_case5(s):
    v = [-s, 1.0, -s, 1.0, 1.0, 1.0, 1.0, 1.0]
    origin = np.array([0.0, 0.0, 0.0])
    direction = np.array([1.0, 1.0, 0.0])

    mesh = lm.known_cases.case5_mesh(v)

    mesh_ts = lm.find_mesh_crossings(mesh, origin, direction)
    tri_ts = lm.find_trilinear_crossings(v, origin, direction)

    status = "CONNECTED" if not tri_ts else "disconnected"
    mesh_status = "connected" if not mesh_ts else "disconnected"
    agree = "agree" if bool(mesh_ts) == bool(tri_ts) else "** DISAGREE **"
    print(f"s={s:.2f}  mesh: {mesh_status} {mesh_ts}  "
          f"trilinear: {status} {tri_ts}  [{agree}]")

    fig = lm.plotting.plot_probe_comparison(mesh, origin, direction, mesh_ts, tri_ts)
    fig.show()


widgets.interact(saddle_case5, s=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05))

## 3. The complementary saddle: case 10

Corners 1 and 3 inside instead of 0 and 2 -- same phenomenon, mirrored.


In [ ]:
def saddle_case10(s):
    v = [1.0, -s, 1.0, -s, 1.0, 1.0, 1.0, 1.0]
    origin = np.array([1.0, 0.0, 0.0])
    direction = np.array([-1.0, 1.0, 0.0])

    mesh = lm.known_cases.case10_mesh(v)

    mesh_ts = lm.find_mesh_crossings(mesh, origin, direction)
    tri_ts = lm.find_trilinear_crossings(v, origin, direction)

    agree = "agree" if bool(mesh_ts) == bool(tri_ts) else "** DISAGREE **"
    print(f"s={s:.2f}  mesh crossings: {mesh_ts}  trilinear crossings: {tri_ts}  [{agree}]")

    fig = lm.plotting.plot_probe_comparison(mesh, origin, direction, mesh_ts, tri_ts)
    fig.show()


widgets.interact(saddle_case10, s=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.05))